Consider a short sentence:
"The cat sat"
Each word is first represented as an embedding vector. These are just numbers that represent the meaning of words. Let's use simple fake embeddings for illustration:

Token	Embedding (3D)
The	    [1, 0, 1]
cat	    [0, 1, 1]
sat	    [1, 1, 0]

These embeddings are passed into the attention layer, which creates Query, Key, and Value vectors for each token via linear transformations.

Creating Query, Key, and Value
Let’s say we apply 3 different linear layers to get:
Token	Query	Key	        Value
The	  [1, 0]	[0.5, 1]	[1, 2]
cat	  [0, 1]	[1, 0.5]	[2, 1]
sat	  [1, 1]	[1, 1]	    [1, 1]
In the forward() method:
    query = self.query_layer(x)
    key = self.key_layer(x)
    value = self.value_layer(x)
Each of these performs a matrix multiplication + bias:
    query = x @ W_q + b_q
    key   = x @ W_k + b_k
    value = x @ W_v + b_v
This is how input embedding (say, [1, 0, 1]) turns into:
a query like [1, 0],
a key like [0.5, 1],
and a value like [1, 2].
The specific numbers in the table ([1, 0], [0.5, 1], [1, 2]) come from learned weights in those linear layers — during training, these weights get adjusted so that attention works effectively.

What Happens in Attention
Each word queries all the other words — including itself — using dot products:
attention_score = Query ⋅ Keyᵀ

Let’s compute this for "cat":
Query("cat") = [0, 1]

It compares with each Key:
Key("The") = [0.5, 1] → Dot = 0×0.5 + 1×1 = 1.0
Key("cat") = [1, 0.5] → Dot = 0×1 + 1×0.5 = 0.5
Key("sat") = [1, 1] → Dot = 0×1 + 1×1 = 1.0
So attention scores for "cat" are: [1.0, 0.5, 1.0]

Then we apply softmax to normalize these scores into probabilities:
softmax([1.0, 0.5, 1.0]) ≈ [0.39, 0.22, 0.39]

Weighted Sum of Values
Now, to get the output for "cat", we take a weighted sum of the Values of all tokens, using the softmax attention weights:
output("cat") = 
0.39 × [1, 2] + 
0.22 × [2, 1] + 
0.39 × [1, 1]
Calculate that:
= [0.39 + 0.44 + 0.39, 0.78 + 0.22 + 0.39] = [1.22, 1.39]

So, the output for "cat" is a new vector [1.22, 1.39], which is a mixture of the Values of all words — weighted by how much attention "cat" paid to them.

